# G2Gs 源码拆解与预测流程

这一份 notebook 是整套教材里最接近“读源码”的部分，但目标不是把源码一行不差地背下来，而是帮助你建立一个清晰的问题框架：

1. G2Gs 为什么要分成两个阶段？
2. 每个阶段分别在预测什么？
3. 一次完整的 retrosynthesis 推理，在代码里是怎样流动的？

如果你在阅读时觉得信息量比较大，不必着急。建议始终抓住一条主线：

**product 先被编码，再预测 center，再变成 synthons，最后一步步生成 reactants。**

---

G2Gs 的总公式可以先记成：

$$
P(R \mid P) = \sum_c P(c \mid P) \cdot P(R \mid S(c, P))
$$

其中：

- `P(c | P)`：先预测 reaction center
- `S(c, P)`：根据 center 把 product 变成 synthons
- `P(R | S)`：再从 synthon 生成 reactant


In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / "teaching_demos").exists() and (candidate / "source_repos").exists():
            return candidate
    raise FileNotFoundError("无法定位项目根目录")


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "teaching_demos/2.single_step_retro_tutorial/2.3.semi-template/2.3.1.g2gs"
CODE_DIR = TUTORIAL_DIR / "code"
DATA_DIR = TUTORIAL_DIR / "data"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print(f"项目根目录: {PROJECT_ROOT}")
print(f"教程目录: {TUTORIAL_DIR}")

import torch
import pandas as pd
from IPython.display import Markdown, display
from g2gs_tutorial import (
    CenterIdentificationModel,
    SynthonCompletionModel,
    build_synthon_completion_dataset,
    draw_reaction_pair_image,
    identify_reaction_center,
    load_demo_reactions,
    molecule_to_graph_tensor,
)

torch.manual_seed(7)


def show_source_block(relative_path, start, end, title):
    path = PROJECT_ROOT / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    snippet = "\n".join(f"{line_no:4d} | {lines[line_no - 1]}" for line_no in range(start, end + 1))
    display(Markdown(f"### {title}\n`{relative_path}:{start}-{end}`"))
    display(Markdown(f"```python\n{snippet}\n```"))


SOURCE_BLOCKS = [
    {
        "stage": "图编码器",
        "source_file": "source_repos/torchdrug/torchdrug/models/gcn.py",
        "symbol": "RelationalGraphConvolutionalNetwork",
        "lines": "88-168",
    },
    {
        "stage": "数据预处理",
        "source_file": "source_repos/torchdrug/torchdrug/datasets/uspto50k.py",
        "symbol": "_get_difference / _get_reaction_center / _get_synthon",
        "lines": "97-220",
    },
    {
        "stage": "中心打分",
        "source_file": "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
        "symbol": "CenterIdentification.predict",
        "lines": "118-154",
    },
    {
        "stage": "中心转 synthon",
        "source_file": "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
        "symbol": "CenterIdentification.predict_synthon",
        "lines": "168-233",
    },
    {
        "stage": "单步动作打分",
        "source_file": "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
        "symbol": "SynthonCompletion._topk_action",
        "lines": "701-788",
    },
    {
        "stage": "beam search 展开",
        "source_file": "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
        "symbol": "SynthonCompletion._apply_action / predict_reactant",
        "lines": "790-920",
    },
    {
        "stage": "总装预测",
        "source_file": "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
        "symbol": "Retrosynthesis.predict",
        "lines": "1090-1158",
    },
]


## 1. 先看源码地图

对于第一次读源码的同学，最容易遇到的问题不是“看不懂某一行”，而是“不知道应该先看哪几段”。

所以这里先给出一张阅读地图，把 G2Gs 的核心源码块按预测流程排好。后面我们就按照这个顺序逐段拆解。


In [ ]:
display(pd.DataFrame(SOURCE_BLOCKS))

## 2. RGCN：公共图编码骨干

无论是第一阶段的中心预测，还是第二阶段的 synthon completion，都先要把分子图编码成向量表示。

在 `torchdrug` 里，这个公共骨干就是 `models.RGCN`。

阅读这一部分时，你不需要纠结每一层卷积的实现细节，先抓住最重要的结论：

- 输入是分子图和节点特征
- 输出是 `node_feature` 和 `graph_feature`
- 后面的任务头都建立在这两个张量之上


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/models/gcn.py",
    88,
    168,
    "RGCN: `RelationalGraphConvolutionalNetwork`",
)


In [ ]:
examples = load_demo_reactions()
example = next(item for item in examples if item.reaction_id == "demo_01")
center = identify_reaction_center(example.reactant_mol, example.product_mol)
product_graph = molecule_to_graph_tensor(example.product_mol, atom_feature_mode="center_identification")

display(draw_reaction_pair_image(example.reactant_mol, example.product_mol, ["Reactant", "Product"]))

center_model = CenterIdentificationModel(
    node_input_dim=product_graph.node_feature.shape[-1],
    edge_input_dim=product_graph.edge_feature.shape[-1],
    num_relation=4,
    num_reaction=10,
    hidden_dim=64,
    num_layers=3,
)
encoder_output = center_model.encoder(product_graph, product_graph.node_feature.float())

display(
    pd.DataFrame(
        [
            {"tensor": "product_graph.node_feature", "shape": tuple(product_graph.node_feature.shape)},
            {"tensor": "encoder_output['node_feature']", "shape": tuple(encoder_output["node_feature"].shape)},
            {"tensor": "encoder_output['graph_feature']", "shape": tuple(encoder_output["graph_feature"].shape)},
        ]
    )
)


从源码和张量形状都能看出一个关键事实：**RGCN 只负责把分子图编码好，并不直接负责输出 reactants。**

把这一点想清楚之后，后面的两阶段结构就会容易很多：

- 编码器负责提供表示
- 任务头负责把表示变成具体预测


## 3. 数据预处理：center 与 synthon 的定义来自哪里

在学习模型之前，先回到数据定义本身。

因为模型到底在“学什么”，其实首先由数据处理代码决定。这里的 `USPTO50k` 源码规定了：

1. reactant 和 product 如何比较差异
2. reaction center 如何被标注出来
3. product 如何被拆成 synthons

如果这一步没有理解透，后面读模型代码时就会很容易迷糊。


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/datasets/uspto50k.py",
    97,
    220,
    "USPTO50k: `_get_difference`, `_get_reaction_center`, `_get_synthon`",
)


In [ ]:
synthon_dataset = build_synthon_completion_dataset([example])
display(
    pd.DataFrame(
        [
            {
                "pair_id": sample["pair_id"],
                "reaction_center": sample["reaction_center"],
                "reactant_smiles": sample["reactant_smiles"],
                "synthon_smiles": sample["synthon_smiles"],
            }
            for sample in synthon_dataset
        ]
    )
)


这一步也是理解“两阶段设计”最自然的入口：

- 第一阶段不是直接预测 reactants，而是先预测 center
- 第二阶段也不是直接接收完整 product，而是接收由 center 生成的 synthons

换句话说，**中间表示不是附属品，而是 G2Gs 设计的核心。**


## 4. 第一阶段：CenterIdentification 源码

现在开始进入真正的预测部分。

读 `CenterIdentification` 时，建议你重点关注三个问题：

1. 输入除了 encoder 输出，还拼接了哪些条件信息？
2. 为什么要同时给节点和边打分？
3. top-k 的 center 候选是怎样转成 synthons 的？

只要把这三个问题读清楚，第一阶段的整体逻辑就已经掌握了。


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
    118,
    154,
    "CenterIdentification: `predict()`",
)


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
    168,
    233,
    "CenterIdentification: `predict_synthon()`",
)


In [ ]:
center_output = center_model(product_graph, reaction=example.reaction_class)
ranked_centers = center_model.rank_centers(product_graph, reaction=example.reaction_class, topk=8)

display(
    pd.DataFrame(
        [
            {"tensor": "node_logits", "shape": tuple(center_output["node_logits"].shape)},
            {"tensor": "edge_logits", "shape": tuple(center_output["edge_logits"].shape)},
            {"tensor": "node_context", "shape": tuple(center_output["node_context"].shape)},
            {"tensor": "graph_context", "shape": tuple(center_output["graph_context"].shape)},
            {
                "tensor": "候选中心总数",
                "shape": product_graph.num_node + product_graph.num_edge // 2,
            },
        ]
    )
)
display(pd.DataFrame(ranked_centers))
print("oracle reaction center:", center["reaction_center"])


第一阶段的预测流程可以概括成 5 个动作：

1. 用 RGCN 编码 product 图
2. 拼接 `reaction / atom / bond / graph` 条件
3. 由 `node head` 和 `edge head` 分别打分
4. 取 top-k reaction center 候选
5. 把高分 center 转成 synthons

因此，第一阶段最准确的理解方式不是“反应物预测器”，而是**反应中心提议器**。


## 5. 第二阶段：SynthonCompletion 的动作空间

第二阶段和很多“直接生成整个分子”的模型不同。它不是一步到位，而是把生成过程拆成一连串小动作。

在源码里，一个动作由四部分组成：

- `node_in`
- `node_out`
- `bond_type`
- `stop`

你可以把它想成：模型每次都在回答“从哪里出发、连到哪里、连什么键、是不是该停了”。


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
    701,
    788,
    "SynthonCompletion: `_topk_action()`",
)


`_topk_action()` 的核心思想，是把“一步编辑”拆成多个小决策分别打分，再把它们组合起来。

这种设计的好处是：

- 每一步决策都更明确
- 动作空间更结构化
- 非常适合配合 beam search 保留多条高分候选路径

所以第二阶段虽然看起来更复杂，但它的逻辑其实很清晰：**把复杂生成任务拆成一串可评分的小动作。**


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
    790,
    920,
    "SynthonCompletion: `_apply_action()` 与 `predict_reactant()`",
)


In [ ]:
first_pair = synthon_dataset[0]
synthon_graph = first_pair["synthon_graph"]

display(pd.DataFrame(first_pair["oracle_actions"]))

synthon_model = SynthonCompletionModel(
    node_input_dim=synthon_graph.node_feature.shape[-1],
    edge_input_dim=synthon_graph.edge_feature.shape[-1],
    num_relation=4,
    num_reaction=10,
    hidden_dim=64,
    num_layers=3,
)

node_context, graph_context = synthon_model._build_node_context(
    synthon_graph,
    reaction_center=tuple(x for x in first_pair["reaction_center"] if x != 0),
    reaction=first_pair["reaction_class"],
)
action_output = synthon_model.score_actions(
    synthon_graph,
    reaction_center=tuple(x for x in first_pair["reaction_center"] if x != 0),
    reaction=first_pair["reaction_class"],
    topk=8,
)

display(
    pd.DataFrame(
        [
            {"tensor": "node_context", "shape": tuple(node_context.shape)},
            {"tensor": "graph_context", "shape": tuple(graph_context.shape)},
            {"tensor": "node_in_logits", "shape": tuple(action_output["node_in_logits"].shape)},
            {"tensor": "stop_logit", "shape": tuple(action_output["stop_logit"].shape)},
        ]
    )
)
display(pd.DataFrame(action_output["top_actions"]))


从源码和教学版张量流对照起来，第二阶段可以总结成一句话：

**先为下一步编辑动作打分，再把动作真正施加到图上，并持续保留多条高分候选路径。**

读到这里时，如果你已经能回答“为什么这里需要 beam search”，就说明第二阶段的核心原理已经掌握了。


## 6. 顶层总装：Retrosynthesis.predict

最后这一节把前面两阶段真正串起来。

你会看到顶层 `predict()` 并没有做新的建模假设，它做的是“流程组织”工作：

1. 调第一阶段生成 synthons
2. 调第二阶段对 synthons 做扩展
3. 合并、排序、去重，得到最终候选 reactants


In [ ]:
show_source_block(
    "source_repos/torchdrug/torchdrug/tasks/retrosynthesis.py",
    1090,
    1158,
    "Retrosynthesis: `predict()`",
)


顶层 `predict()` 可以帮助我们把 G2Gs 的整体思想再说完整一遍：

1. 先预测 product 应该从哪里断开
2. 再把断开的片段逐步补成 reactants
3. 最后对多条候选路径统一排序

所以 G2Gs 的本质不是“单次分类”或“单次生成”，而是：

**图编码 + 中间结构预测 + 条件搜索**


In [ ]:
pipeline_summary = pd.DataFrame(
    [
        {
            "step": "1. Encode product graph",
            "source_symbol": "models.RGCN.forward",
            "intermediate": "node_feature / graph_feature",
            "teaching_mapping": "SimpleRGCNEncoder",
        },
        {
            "step": "2. Score reaction centers",
            "source_symbol": "CenterIdentification.predict",
            "intermediate": "node_logits / edge_logits",
            "teaching_mapping": "CenterIdentificationModel",
        },
        {
            "step": "3. Convert center to synthons",
            "source_symbol": "CenterIdentification.predict_synthon",
            "intermediate": "synthon batch",
            "teaching_mapping": "identify_reaction_center + build_synthon_completion_dataset",
        },
        {
            "step": "4. Score edit actions",
            "source_symbol": "SynthonCompletion._topk_action",
            "intermediate": "node_in / node_out / bond / stop",
            "teaching_mapping": "SynthonCompletionModel.score_actions",
        },
        {
            "step": "5. Expand beams and rank reactants",
            "source_symbol": "predict_reactant + Retrosynthesis.predict",
            "intermediate": "top-k reactant sets",
            "teaching_mapping": "源码块 + 最小张量流对照",
        },
    ]
)
display(pipeline_summary)


## 7. 建议的阅读顺序

如果你是第一次系统学习 G2Gs，我建议按下面的顺序反复阅读这份 notebook：

1. 先回到数据预处理，理解 center 和 synthons 是怎样定义的。
2. 再读第一阶段，理解为什么要做 node / edge 联合打分。
3. 最后读第二阶段和顶层流程，理解为什么需要 beam search。

不要追求一遍读完就记住所有细节。对学生来说，更重要的是先建立整体框架，再慢慢补齐每个模块的局部实现。
